# ICBHI AST-SAM — Colab Training Notebook

**Workflow:**
1. Mount Google Drive (persistent checkpoints survive disconnects)
2. Clone / pull your GitHub repo
3. Install dependencies
4. Preprocess data (once) - automatically downloads dataset if needed
5. Train with `--resume` so you pick up from the last saved epoch
6. Evaluate

> **Auto-download strategy:** The `scripts/preprocess.py` script automatically downloads the ICBHI dataset if it's not found. This eliminates manual uploads and works seamlessly whether you're training locally or in Colab.

In [ ]:
# ── Verify GPU ────────────────────────────────────────────────
import torch
print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# All persistent files live here — survives runtime resets
DRIVE_PROJECT = '/content/drive/MyDrive/ICBHI-AST-SAM'
os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_PROJECT}/results',     exist_ok=True)
print(f'Drive project folder: {DRIVE_PROJECT}')

In [ ]:
# ── Clone / pull repo ─────────────────────────────────────────
REPO_URL = 'https://github.com/ajammoussi/SAM-Optimized-AST-Respiratory-Sound-Classification'
REPO_DIR = '/content/ICBHI-AST-SAM'

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

%cd $REPO_DIR
!ls -la

In [ ]:
# ── Verify dataset / split file and wire the first valid copy ──
import os
import subprocess

REPO_DIR = '/content/ICBHI-AST-SAM'
DRIVE_PROJECT = '/content/drive/MyDrive/ICBHI-AST-SAM'
REPO_DATA_DIR = f'{REPO_DIR}/data'
DRIVE_DATA_DIR = f'{DRIVE_PROJECT}/data'

print('Checking for ICBHI dataset...\n')

# Let download_data.py handle everything (check, download, verify)
result = subprocess.run(
    ['python', 'scripts/download_data.py', '--data_dir', DRIVE_DATA_DIR],
    cwd=REPO_DIR,
    capture_output=False,
)

if result.returncode != 0:
    raise RuntimeError('Failed to setup dataset. Check the output above.')

# Create symlink to Drive dataset (persistent across Colab sessions)
if os.path.exists(REPO_DATA_DIR):
    if os.path.islink(REPO_DATA_DIR):
        os.unlink(REPO_DATA_DIR)
    elif os.path.isdir(REPO_DATA_DIR):
        import shutil
        shutil.rmtree(REPO_DATA_DIR)

os.symlink(DRIVE_DATA_DIR, REPO_DATA_DIR)
print(f'\n✓ Linked {REPO_DATA_DIR} → {DRIVE_DATA_DIR}')

In [ ]:
# ── (Optional): VS Code Remote Tunnel ──────────────────────────
# This lets you edit files from your local VS Code while the GPU runs here.
# Requires: VS Code + "Remote - Tunnels" extension installed locally.
#
# After running this cell:
#   1. Copy the GitHub auth code that appears in the output.
#   2. Go to https://github.com/login/device and paste it.
#   3. In VS Code: Ctrl+Shift+P → 'Remote Tunnels: Connect to Tunnel'
#      → select the tunnel name you set below.

TUNNEL_NAME = 'colab-icbhi'   # change to anything memorable

!pip install vscode-colab -q
import vscode_colab
vscode_colab.connect(
    name=TUNNEL_NAME,
    git_user_name='Your Name',          # optional, for git commits
    git_user_email='you@example.com',   # optional
)

In [ ]:
# ── Install dependencies ──────────────────────────────────────
!pip install -r requirements.txt -q

In [ ]:
# ── Symlink checkpoints & results to Drive ────────────────────
# This means scripts/train.py writes directly to Drive without any manual copying.
import subprocess

LOCAL_CKPT    = f'{REPO_DIR}/checkpoints'
LOCAL_RESULTS = f'{REPO_DIR}/results'
DRIVE_CKPT    = f'{DRIVE_PROJECT}/checkpoints'
DRIVE_RESULTS = f'{DRIVE_PROJECT}/results'

for local, drive_path in [(LOCAL_CKPT, DRIVE_CKPT), (LOCAL_RESULTS, DRIVE_RESULTS)]:
    if os.path.islink(local):
        os.remove(local)
    elif os.path.isdir(local):
        import shutil; shutil.rmtree(local)
    os.symlink(drive_path, local)
    print(f'Linked {local} → {drive_path}')

In [ ]:
# ── Preprocess (run ONCE, skip if .npz already exists on Drive) ─
NPZ_PATH = f'{DRIVE_PROJECT}/icbhi_ast_16k_8s_spectrograms.npz'

if os.path.exists(NPZ_PATH):
    print(f'Pre-computed spectrograms found: {NPZ_PATH}  — skipping preprocessing.')
else:
    print('Running preprocessing (this takes ~5-10 min) …')
    # scripts/preprocess.py auto-detects data from ./data/ (which is symlinked to Drive)
    !python scripts/preprocess.py --output $NPZ_PATH

# Symlink the npz into the repo dir so scripts/train.py finds it at its default path
LOCAL_NPZ = f'{REPO_DIR}/icbhi_ast_16k_8s_spectrograms.npz'
if not os.path.exists(LOCAL_NPZ):
    os.symlink(NPZ_PATH, LOCAL_NPZ)
    print(f'Linked {LOCAL_NPZ} → {NPZ_PATH}')

In [ ]:
# ── Train  (add --resume to continue after a disconnect) ───────
# Remove --resume on the very first run.
!python scripts/train.py \
    --data_path    ./icbhi_ast_16k_8s_spectrograms.npz \
    --checkpoint_dir ./checkpoints \
    --results_dir  ./results \
    --epochs       20 \
    --batch_size   24 \
    --lr           1e-5 \
    --looksam_k    2 \
    --num_workers  4 \
    --scheduler_t0 10 \
    --amp_dtype    fp16 \
    --resume

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────
!python scripts/evaluate.py \
    --data_path  ./icbhi_ast_16k_8s_spectrograms.npz \
    --model_path ./checkpoints/best_model.pth \
    --output_dir ./results

In [ ]:
# ── Inline preview of figures ────────────────────────────────
from IPython.display import display, Image
import glob

for fig_path in sorted(glob.glob('./results/*.png')):
    print(fig_path)
    display(Image(fig_path, width=800))